### Chapter 5.3: Complete Anatomy를 넘어서 - 3D 인체 모델
### Total Segmentator를 활용한 개인화된 3D 모델 생성

### Chapter 5.3: AI로 구현하는 개인화된 3D 인체 모델
#
### Total Segmentator로 59개 해부학적 구조 자동 분할

##### 1. 라이브러리 설치


In [ ]:
# !pip install TotalSegmentator nibabel numpy matplotlib vtk SimpleITK

In [ ]:
import os
import subprocess
from pathlib import Path
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import SimpleITK as sitk

print("✅ 라이브러리 임포트 완료")

##### 2. Total Segmentator 설치 확인


In [ ]:
# Total Segmentator 설치 확인
try:
    import totalsegmentator
    print("✅ TotalSegmentator가 설치되어 있습니다.")
    print(f"버전: {totalsegmentator.__version__}")
except ImportError:
    print("TotalSegmentator 설치 중...")
    !pip install TotalSegmentator
    print("✅ 설치 완료!")

##### 3. 샘플 데이터 다운로드


In [ ]:
# 데이터 디렉토리
DATA_DIR = './data/total_segmentator'
os.makedirs(DATA_DIR, exist_ok=True)

print(f"데이터 경로: {DATA_DIR}")

In [ ]:
# 샘플 CT 다운로드 (GitHub)
import urllib.request

sample_url = "https://github.com/wasserth/TotalSegmentator/raw/master/resources/test_image.nii.gz"
sample_path = os.path.join(DATA_DIR, 'test_ct.nii.gz')

if not os.path.exists(sample_path):
    print(f"📥 샘플 CT 다운로드 중...")
    try:
        urllib.request.urlretrieve(sample_url, sample_path)
        print(f"✅ 다운로드 완료: {sample_path}")
    except Exception as e:
        print(f"❌ 다운로드 실패: {e}")
        print("💡 수동 다운로드: https://github.com/wasserth/TotalSegmentator/tree/master/resources")
else:
    print(f"✅ 샘플 데이터 존재: {sample_path}")

##### 4. Total Segmentator 실행


In [ ]:
# 출력 디렉토리
OUTPUT_DIR = os.path.join(DATA_DIR, 'segmentation_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"출력 경로: {OUTPUT_DIR}")

In [ ]:
# Total Segmentator 실행 (CLI)
if os.path.exists(sample_path):
    print("🔬 Total Segmentator 실행 중...")
    print("(약 2-5분 소요)")

    cmd = f'TotalSegmentator -i "{sample_path}" -o "{OUTPUT_DIR}" --fast'

    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if result.returncode == 0:
        print("✅ 세그멘테이션 완료!")
        print(f"📁 결과 저장: {OUTPUT_DIR}")
    else:
        print(f"❌ 실행 실패")
        print(f"에러: {result.stderr}")
else:
    print("⚠️  샘플 데이터가 없습니다.")

##### 5. 결과 확인


In [ ]:
# 생성된 파일 목록
output_files = list(Path(OUTPUT_DIR).glob('*.nii.gz'))

print(f"✅ 생성된 파일: {len(output_files)}개")
print("\n주요 장기:")

# 주요 장기 목록
major_organs = [
    'liver', 'spleen', 'kidney_right', 'kidney_left',
    'heart', 'lung_upper_lobe_left', 'lung_upper_lobe_right'
]

for organ in major_organs:
    organ_file = Path(OUTPUT_DIR) / f'{organ}.nii.gz'
    if organ_file.exists():
        print(f"  ✓ {organ}")

##### 6. CT 원본 데이터 로드


In [ ]:
# 원본 CT 로드
ct_nii = nib.load(sample_path)
ct_volume = ct_nii.get_fdata()

print(f"CT 볼륨 크기: {ct_volume.shape}")
print(f"복셀 간격: {ct_nii.header.get_zooms()}")

##### 7. 세그멘테이션 결과 로드 및 시각화


In [ ]:
# 간(liver) 세그멘테이션 로드
liver_file = Path(OUTPUT_DIR) / 'liver.nii.gz'

if liver_file.exists():
    liver_nii = nib.load(liver_file)
    liver_mask = liver_nii.get_fdata()

    print(f"간 마스크 크기: {liver_mask.shape}")
    print(f"간 복셀 수: {np.sum(liver_mask > 0)}")
else:
    print("⚠️  간 세그멘테이션 파일이 없습니다.")
    liver_mask = None

In [ ]:
# 중간 슬라이스 시각화
if liver_mask is not None:
    middle_slice = ct_volume.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 원본 CT
    axes[0].imshow(ct_volume[:, :, middle_slice].T, cmap='gray', origin='lower')
    axes[0].set_title('원본 CT')
    axes[0].axis('off')

    # 세그멘테이션
    axes[1].imshow(liver_mask[:, :, middle_slice].T, cmap='jet', origin='lower')
    axes[1].set_title('간 세그멘테이션')
    axes[1].axis('off')

    # 오버레이
    axes[2].imshow(ct_volume[:, :, middle_slice].T, cmap='gray', origin='lower')
    masked_liver = np.ma.masked_where(
        liver_mask[:, :, middle_slice].T == 0,
        liver_mask[:, :, middle_slice].T
    )
    axes[2].imshow(masked_liver, cmap='jet', alpha=0.5, origin='lower')
    axes[2].set_title('오버레이')
    axes[2].axis('off')

    plt.tight_layout()
    plt.savefig('liver_segmentation.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("시각화 저장: liver_segmentation.png")

##### 8. 다중 장기 시각화


In [ ]:
# 여러 장기 동시 표시
organs_to_show = ['liver', 'spleen', 'kidney_right', 'kidney_left']
colors = ['red', 'blue', 'green', 'yellow']

# 복합 마스크 생성
combined_mask = np.zeros_like(ct_volume[:, :, middle_slice])

for i, organ in enumerate(organs_to_show, start=1):
    organ_file = Path(OUTPUT_DIR) / f'{organ}.nii.gz'
    if organ_file.exists():
        organ_nii = nib.load(organ_file)
        organ_mask = organ_nii.get_fdata()
        organ_slice = organ_mask[:, :, middle_slice]
        combined_mask[organ_slice > 0] = i

In [ ]:
# 시각화
fig, ax = plt.subplots(figsize=(10, 10))

ax.imshow(ct_volume[:, :, middle_slice].T, cmap='gray', origin='lower')

masked_combined = np.ma.masked_where(combined_mask.T == 0, combined_mask.T)
ax.imshow(masked_combined, cmap=ListedColormap(colors), alpha=0.5, origin='lower')

# 범례
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=colors[i], label=organ)
    for i, organ in enumerate(organs_to_show)
]
ax.legend(handles=legend_elements, loc='upper right')

ax.set_title(f'다중 장기 세그멘테이션 (Slice {middle_slice})')
ax.axis('off')

plt.savefig('multi_organ_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

print("📊 다중 장기 시각화 저장: multi_organ_segmentation.png")

##### 9. 부피 계산


In [ ]:
# 장기 부피 계산
voxel_spacing = ct_nii.header.get_zooms()  # (x, y, z) mm
voxel_volume_cm3 = np.prod(voxel_spacing) / 1000.0  # mm³ → cm³

print("📊 장기별 부피 분석")
print("="*50)

for organ in organs_to_show:
    organ_file = Path(OUTPUT_DIR) / f'{organ}.nii.gz'
    if organ_file.exists():
        organ_nii = nib.load(organ_file)
        organ_mask = organ_nii.get_fdata()

        num_voxels = np.sum(organ_mask > 0)
        volume_cm3 = num_voxels * voxel_volume_cm3

        print(f"{organ:20s}: {volume_cm3:8.2f} cm³")

print("="*50)